# Fase 2 — Aprendizaje de Máquina
## Clasificación de Género en Twitter · SVM · Random Forest · Gradient Boosting · Stacking

**Curso:** Sistemas Inteligentes · Pontificia Universidad Javeriana  
**Módulo:** Aprendizaje de Máquina  

---
### Moralejas aplicadas desde los papers:
- **Paper 1 — Vashisth et al. (2021) IEEE ICCMC:** RF y Gradient Boosting superan a LR/MLP; añadir **meta-features** lingüísticos (longitud, puntuación, emojis) sobre TF-IDF mejora el desempeño → *aplicado en Feature Engineering (Sección 2)*
- **Paper 2 — Yang et al. (2021) JAMIA Open:** Un **meta-clasificador (Stacking)** que combina SVM + RF + GB logra mejor desempeño que cualquier modelo individual → *aplicado directamente como modelo final (Sección 4)*
- **Paper 3 — Ensemble DL (2020) arXiv:** Combinar múltiples **espacios de features** (word n-grams + char n-grams + metadatos) eleva el desempeño independiente del clasificador → *aplicado en la construcción de features (Sección 2)*

## 0 · Instalación de dependencias

In [ ]:
# Descomentar y correr solo la primera vez
# !pip install gradio scikit-learn pandas numpy matplotlib seaborn wordcloud

## 1 · Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
from scipy.sparse import hstack, csr_matrix
import joblib

print('✅ Librerías cargadas')

In [ ]:
DATA_PATH  = '../data/gender-classifier-clean.csv'
TEXT_COL   = 'text'
LABEL_COL  = 'gender'

df = pd.read_csv(DATA_PATH)

# Filtrar solo male/female — excluir 'brand' (no es clasificación de género)
df = df[df[LABEL_COL].isin(['male', 'female'])].copy()
df['label'] = df[LABEL_COL].map({'female': 0, 'male': 1})
df = df.dropna(subset=[TEXT_COL, 'label'])
df[TEXT_COL] = df[TEXT_COL].astype(str)

X_text = df[TEXT_COL].values
y      = df['label'].values

print(f'Dataset listo: {len(X_text)} muestras (brand excluido)')
print(f'Balance: {(y==0).sum()} female ({(y==0).mean()*100:.1f}%) | {(y==1).sum()} male ({(y==1).mean()*100:.1f}%)')
df.head(3)

## 2 · Feature Engineering
**Moraleja Paper 1 (Vashisth) + Paper 3 (Ensemble DL):** Combinar TF-IDF word, char n-grams y meta-features supera a cualquier espacio de features individual.

In [ ]:
def extract_meta_features(texts):
    """12 meta-features lingüísticos inspirados en Vashisth et al. (2021)"""
    features = []
    for text in texts:
        text = str(text)
        words = text.split()
        n_chars    = len(text)
        n_words    = len(words)
        n_upper    = sum(1 for c in text if c.isupper())
        n_punct    = sum(1 for c in text if c in '!?.,;:')
        n_excl     = text.count('!')
        n_question = text.count('?')
        n_hashtag  = text.count('#')
        n_mention  = text.count('@')
        n_emoji    = sum(1 for c in text if ord(c) > 127)
        avg_wlen   = np.mean([len(w) for w in words]) if words else 0
        up_ratio   = n_upper / max(n_chars, 1)
        pu_ratio   = n_punct / max(n_chars, 1)
        features.append([n_chars, n_words, n_upper, n_punct, n_excl,
                         n_question, n_hashtag, n_mention, n_emoji,
                         avg_wlen, up_ratio, pu_ratio])
    cols = ['n_chars','n_words','n_upper','n_punct','n_exclamation',
            'n_question','n_hashtag','n_mention','n_emoji',
            'avg_word_len','upper_ratio','punct_ratio']
    return pd.DataFrame(features, columns=cols)

META_FEAT_NAMES = ['n_chars','n_words','n_upper','n_punct','n_exclamation',
                   'n_question','n_hashtag','n_mention','n_emoji',
                   'avg_word_len','upper_ratio','punct_ratio']

meta_df = extract_meta_features(X_text)
meta_df['label'] = y
print('Meta-features extraídos:')
meta_df.describe().round(3)

In [ ]:
# Visualización: diferencias por género en meta-features
feats_plot = ['n_words','n_exclamation','n_emoji','n_hashtag',
              'n_mention','upper_ratio','punct_ratio','avg_word_len']
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, feat in zip(axes.flat, feats_plot):
    for lbl, color, name in [(0,'#E76F51','Female'),(1,'#264653','Male')]:
        data = meta_df[meta_df['label']==lbl][feat].dropna()
        ax.hist(data, bins=30, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)
plt.suptitle('Distribución de Meta-Features por Género  (Vashisth et al. 2021)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('meta_features_dist.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ meta_features_dist.png guardado')

In [ ]:
# Split estratificado 80/20
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF word-level (bigramas)
tfidf_word = TfidfVectorizer(max_features=8000, ngram_range=(1,2),
                              min_df=3, sublinear_tf=True)
# TF-IDF char-level (capta patrones ortográficos de género)
tfidf_char = TfidfVectorizer(max_features=3000, ngram_range=(2,4),
                              analyzer='char_wb', sublinear_tf=True)

Xw_tr = tfidf_word.fit_transform(X_train_txt)
Xw_te = tfidf_word.transform(X_test_txt)
Xc_tr = tfidf_char.fit_transform(X_train_txt)
Xc_te = tfidf_char.transform(X_test_txt)

meta_tr = csr_matrix(extract_meta_features(X_train_txt).values)
meta_te = csr_matrix(extract_meta_features(X_test_txt).values)

# Feature matrix final: word + char + meta
X_train = hstack([Xw_tr, Xc_tr, meta_tr])
X_test  = hstack([Xw_te, Xc_te, meta_te])

print(f'✅ Train: {X_train.shape} | Test: {X_test.shape}')
print(f'   {tfidf_word.max_features} word features + {tfidf_char.max_features} char features + {len(META_FEAT_NAMES)} meta-features')

## 3 · Entrenamiento de los 4 modelos

| Modelo | Técnica | Paper que lo justifica |
|--------|---------|----------------------|
| SVM (Linear) | Máquinas de Vectores de Soporte | Paper 2 — Yang 2021 |
| Random Forest | Bagging Ensemble | Paper 1 — Vashisth 2021 |
| Gradient Boosting | Boosting Ensemble | Paper 1 — Vashisth 2021 |
| Stacking | Meta-clasificador (RF+GB+SVM → LR) | Paper 2 — Yang 2021 |

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
trained_models = {}   # guarda los modelos listos
results_list   = []   # guarda métricas

def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    """Entrena, evalúa y guarda resultados."""
    model.fit(X_tr, y_tr)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    acc  = accuracy_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred, average='weighted')
    prec = precision_score(y_te, y_pred, average='weighted')
    rec  = recall_score(y_te, y_pred, average='weighted')
    auc  = roc_auc_score(y_te, y_proba)
    trained_models[name] = model
    results_list.append({'Modelo': name, 'Accuracy': round(acc,4),
                         'Precision': round(prec,4), 'Recall': round(rec,4),
                         'F1-Weighted': round(f1,4), 'AUC-ROC': round(auc,4),
                         'Fase': 'Fase 2 (AM)'})
    print(f'\n── {name} ──')
    print(f'   Accuracy: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}')
    return model

print('Funciones listas. Corriendo modelos...')

In [ ]:
# ── 3.1  SVM (Linear + calibración para predict_proba) ──────────────────────
# LinearSVC es óptimo para TF-IDF de alta dimensión
# CalibratedClassifierCV añade predict_proba necesario para ROC y Stacking
svm_base = CalibratedClassifierCV(
    LinearSVC(C=1.0, max_iter=2000, random_state=42), cv=3
)
evaluate('SVM (Linear)', svm_base, X_train, X_test, y_train, y_test)
print(classification_report(y_test, trained_models['SVM (Linear)'].predict(X_test),
                             target_names=['Female','Male']))

In [ ]:
# ── 3.2  Random Forest + GridSearch ─────────────────────────────────────────
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth'   : [None, 20, 30],
    'max_features': ['sqrt', 'log2']
}
rf_grid = GridSearchCV(
    RandomForestClassifier(n_jobs=-1, random_state=42),
    rf_params, cv=cv, scoring='f1_weighted', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)
print(f'Mejor config RF: {rf_grid.best_params_}  |  F1 CV: {rf_grid.best_score_:.4f}')
evaluate('Random Forest', rf_grid.best_estimator_, X_train, X_test, y_train, y_test)
print(classification_report(y_test, trained_models['Random Forest'].predict(X_test),
                             target_names=['Female','Male']))

In [ ]:
# ── 3.3  Gradient Boosting + GridSearch ─────────────────────────────────────
gb_params = {
    'n_estimators' : [100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth'    : [3, 5]
}
gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_params, cv=cv, scoring='f1_weighted', n_jobs=-1, verbose=1
)
gb_grid.fit(X_train.toarray() if hasattr(X_train,'toarray') else X_train, y_train)
print(f'Mejor config GB: {gb_grid.best_params_}  |  F1 CV: {gb_grid.best_score_:.4f}')

# Wrapper para que predict_proba funcione con sparse
class DenseGB:
    def __init__(self, model):
        self.model = model
    def fit(self, X, y):
        self.model.fit(X.toarray() if hasattr(X,'toarray') else X, y)
        return self
    def predict(self, X):
        return self.model.predict(X.toarray() if hasattr(X,'toarray') else X)
    def predict_proba(self, X):
        return self.model.predict_proba(X.toarray() if hasattr(X,'toarray') else X)

gb_model = DenseGB(gb_grid.best_estimator_)
evaluate('Gradient Boosting', gb_model, X_train, X_test, y_train, y_test)
print(classification_report(y_test, trained_models['Gradient Boosting'].predict(X_test),
                             target_names=['Female','Male']))

In [ ]:
# ── 3.4  Stacking (Paper 2 — Yang 2021) ────────────────────────────────────
# Base learners: SVM + RF + GB  →  Meta-clasificador: Logistic Regression
# Moraleja: el meta-clasificador aprende cuándo confiar en cada modelo base

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler

class DenseTransformer:
    def fit(self, X, y=None): return self
    def transform(self, X): return X.toarray() if hasattr(X,'toarray') else X
    def fit_transform(self, X, y=None): return self.transform(X)

estimators = [
    ('svm', CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000, random_state=42), cv=3)),
    ('rf',  rf_grid.best_estimator_),
    ('gb',  Pipeline([
                ('dense', DenseTransformer()),
                ('gb',    gb_grid.best_estimator_)
            ]))
]

stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(C=1.0, max_iter=500, random_state=42),
    cv=5,
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False
)
evaluate('Stacking (SVM+RF+GB→LR)', stacking, X_train, X_test, y_train, y_test)
print(classification_report(y_test, trained_models['Stacking (SVM+RF+GB→LR)'].predict(X_test),
                             target_names=['Female','Male']))

## 4 · Protocolo Experimental y Análisis de Resultados

In [ ]:
# Resultados de fases anteriores — ACTUALIZAR con valores reales del repo
prev_results = [
    {'Modelo':'Logistic Regression', 'Accuracy':0.00,'Precision':0.00,'Recall':0.00,
     'F1-Weighted':0.00,'AUC-ROC':0.00,'Fase':'Prev'},
    {'Modelo':'Perceptron',          'Accuracy':0.00,'Precision':0.00,'Recall':0.00,
     'F1-Weighted':0.00,'AUC-ROC':0.00,'Fase':'Prev'},
    {'Modelo':'MLP (manual)',         'Accuracy':0.00,'Precision':0.00,'Recall':0.00,
     'F1-Weighted':0.00,'AUC-ROC':0.00,'Fase':'Prev'},
    {'Modelo':'MLP + DE (Fase 1)',    'Accuracy':0.00,'Precision':0.00,'Recall':0.00,
     'F1-Weighted':0.00,'AUC-ROC':0.00,'Fase':'Fase 1 (EA)'},
]

df_results = pd.DataFrame(prev_results + results_list)
print('=== Tabla Comparativa General ===')
print(df_results[['Modelo','Accuracy','F1-Weighted','AUC-ROC','Fase']].to_string(index=False))

In [ ]:
# ── Curvas ROC de los 4 modelos nuevos ──────────────────────────────────────
colors_roc = {'SVM (Linear)':'#264653','Random Forest':'#2A9D8F',
              'Gradient Boosting':'#E9C46A','Stacking (SVM+RF+GB→LR)':'#E76F51'}

fig, ax = plt.subplots(figsize=(8, 6))
for name, model in trained_models.items():
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, lw=2.5, color=colors_roc[name], label=f'{name}  (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--', lw=1, alpha=0.4, label='Random')
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title('Curvas ROC — Fase 2: Aprendizaje de Máquina', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.spines[['top','right']].set_visible(False)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ roc_curves.png guardado')

In [ ]:
# ── Comparativa general (todos los modelos del proyecto) ────────────────────
df_plot = df_results[df_results['Accuracy'] > 0].copy()
x = np.arange(len(df_plot))
w = 0.28

palette = ['#B0BEC5' if f=='Prev' else '#E9C46A' if f=='Fase 1 (EA)' else '#E76F51'
           for f in df_plot['Fase']]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# F1 y Accuracy
axes[0].bar(x - w/2, df_plot['Accuracy'],    w, label='Accuracy',    color=palette, alpha=0.85, edgecolor='white')
axes[0].bar(x + w/2, df_plot['F1-Weighted'], w, label='F1-Weighted', color=palette, alpha=0.55, edgecolor='white', hatch='//')
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_plot['Modelo'], rotation=35, ha='right', fontsize=9)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Accuracy & F1 por Modelo', fontweight='bold')
axes[0].spines[['top','right']].set_visible(False)
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend()

# AUC-ROC
axes[1].bar(x, df_plot['AUC-ROC'], color=palette, edgecolor='white', alpha=0.85)
axes[1].axhline(0.5, color='red', ls='--', alpha=0.4, label='Random')
axes[1].set_xticks(x)
axes[1].set_xticklabels(df_plot['Modelo'], rotation=35, ha='right', fontsize=9)
axes[1].set_ylim(0, 1.1)
axes[1].set_title('AUC-ROC por Modelo', fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)
axes[1].grid(axis='y', alpha=0.3)
axes[1].legend()

legend_patches = [
    mpatches.Patch(color='#B0BEC5', label='Modelos previos'),
    mpatches.Patch(color='#E9C46A', label='Fase 1 (EA)'),
    mpatches.Patch(color='#E76F51', label='Fase 2 (AM)'),
]
fig.legend(handles=legend_patches, loc='upper center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, 1.04), framealpha=0.9)
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ model_comparison.png guardado')

In [ ]:
# ── Feature Importance (Random Forest) ──────────────────────────────────────
word_names = tfidf_word.get_feature_names_out().tolist()
char_names = [f'char_{f}' for f in tfidf_char.get_feature_names_out()]
all_names  = word_names + char_names + META_FEAT_NAMES

importances = trained_models['Random Forest'].feature_importances_
feat_df = pd.DataFrame({'feature': all_names, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=False).reset_index(drop=True)

COLOR_MAP = {'TF-IDF Word':'#264653','TF-IDF Char':'#2A9D8F','Meta-Feature':'#E76F51'}

def feat_type(name):
    if name in META_FEAT_NAMES: return 'Meta-Feature'
    if name.startswith('char_'): return 'TF-IDF Char'
    return 'TF-IDF Word'

top25 = feat_df.head(25).copy()
top25['type'] = top25['feature'].apply(feat_type)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(range(25), top25['importance'].values,
        color=[COLOR_MAP[t] for t in top25['type']], edgecolor='white', height=0.7)
ax.set_yticks(range(25))
ax.set_yticklabels(top25['feature'].values, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Importancia (Mean Decrease Impurity)')
ax.set_title('Top 25 Features — Random Forest', fontweight='bold', fontsize=12)
legend_el = [mpatches.Patch(facecolor=v, label=k) for k,v in COLOR_MAP.items()]
ax.legend(handles=legend_el, loc='lower right')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ feature_importance.png guardado')

In [ ]:
# ── Matrices de confusión de los 4 modelos ───────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, model) in zip(axes, trained_models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
                xticklabels=['Female','Male'], yticklabels=['Female','Male'],
                linewidths=0.5, cbar=False)
    ax.set_title(name, fontsize=9, fontweight='bold')
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')
plt.suptitle('Matrices de Confusión — Fase 2 AM', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ confusion_matrices.png guardado')

## 5 · Guardar modelos

In [ ]:
joblib.dump(trained_models, 'all_models.pkl')
joblib.dump(tfidf_word,     'tfidf_word.pkl')
joblib.dump(tfidf_char,     'tfidf_char.pkl')
df_results.to_csv('resultados_comparativos.csv', index=False)
feat_df.to_csv('feature_importances.csv', index=False)
print('✅ Modelos y CSVs guardados')

---
## 6 · 🎛️ Dashboard Interactivo — Gradio
**4 tabs:** Clasificar tweet (elige modelo) · Comparar todos los modelos · Explorar features · Análisis experimental

In [ ]:
import gradio as gr
from wordcloud import WordCloud

MODEL_NAMES = list(trained_models.keys())

# ── Predicción con modelo seleccionable ─────────────────────────────────────
def predict_gender(tweet_text, model_name):
    if not tweet_text.strip():
        return '⚠️ Ingresa un tweet', None, ''
    model = trained_models[model_name]
    wf = tfidf_word.transform([tweet_text])
    cf = tfidf_char.transform([tweet_text])
    mf = csr_matrix(extract_meta_features([tweet_text]).values)
    X_in = hstack([wf, cf, mf])
    proba = model.predict_proba(X_in)[0]
    pred  = model.predict(X_in)[0]
    label = '♀️ Femenino' if pred == 0 else '♂️ Masculino'
    conf  = proba[pred] * 100

    # Top palabras del tweet que están en el vocabulario
    words = [w.lower() for w in tweet_text.split()]
    top_act = feat_df[feat_df['feature'].isin(words)].head(5)
    feat_info = top_act[['feature','importance']].to_string(index=False) if not top_act.empty else 'N/A'

    fig, ax = plt.subplots(figsize=(5, 2))
    ax.barh(['Femenino','Masculino'], [proba[0]*100, proba[1]*100],
            color=['#E76F51','#264653'], height=0.5)
    ax.set_xlim(0, 100)
    ax.set_xlabel('Probabilidad (%)')
    ax.set_title(f'{label} — {conf:.1f}% ({model_name})', fontweight='bold')
    for i, val in enumerate([proba[0], proba[1]]):
        ax.text(val*100 + 1, i, f'{val*100:.1f}%', va='center', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()

    return f'{label} — Confianza: {conf:.1f}%', fig, f'Top features activos:\n{feat_info}'

# ── Comparación de modelos ───────────────────────────────────────────────────
def show_comparison():
    df_p = df_results[df_results['Accuracy'] > 0].copy()
    x = np.arange(len(df_p))
    w = 0.28
    palette = ['#B0BEC5' if f=='Prev' else '#E9C46A' if f=='Fase 1 (EA)' else '#E76F51'
               for f in df_p['Fase']]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(x-w/2, df_p['Accuracy'],    w, color=palette, alpha=0.85, label='Accuracy', edgecolor='w')
    axes[0].bar(x+w/2, df_p['F1-Weighted'], w, color=palette, alpha=0.55, label='F1', edgecolor='w', hatch='//')
    axes[0].set_xticks(x); axes[0].set_xticklabels(df_p['Modelo'], rotation=35, ha='right', fontsize=8)
    axes[0].set_ylim(0,1.1); axes[0].set_title('Accuracy & F1', fontweight='bold')
    axes[0].legend(); axes[0].spines[['top','right']].set_visible(False); axes[0].grid(axis='y', alpha=0.3)
    axes[1].bar(x, df_p['AUC-ROC'], color=palette, edgecolor='w', alpha=0.85)
    axes[1].axhline(0.5, color='red', ls='--', alpha=0.4)
    axes[1].set_xticks(x); axes[1].set_xticklabels(df_p['Modelo'], rotation=35, ha='right', fontsize=8)
    axes[1].set_ylim(0,1.1); axes[1].set_title('AUC-ROC', fontweight='bold')
    axes[1].spines[['top','right']].set_visible(False); axes[1].grid(axis='y', alpha=0.3)
    plt.tight_layout()
    return fig

# ── WordCloud por género ─────────────────────────────────────────────────────
def show_wordcloud(gender):
    lv = 0 if gender == 'Femenino' else 1
    texts = df[df['label']==lv][TEXT_COL].astype(str).tolist()
    wc = WordCloud(width=700, height=380, background_color='white',
                   colormap='Reds' if lv==0 else 'Blues',
                   max_words=80, collocations=False).generate(' '.join(texts))
    fig, ax = plt.subplots(figsize=(9,5))
    ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
    ax.set_title(f'Vocabulario más frecuente — {gender}', fontweight='bold', fontsize=13)
    plt.tight_layout()
    return fig

# ── Top features ─────────────────────────────────────────────────────────────
def show_top_features(n):
    n = int(n)
    top = feat_df.head(n).copy()
    top['type'] = top['feature'].apply(feat_type)
    fig, ax = plt.subplots(figsize=(9, n*0.38+1))
    ax.barh(range(n), top['importance'].values[::-1],
            color=[COLOR_MAP[t] for t in top['type']][::-1], edgecolor='white', height=0.7)
    ax.set_yticks(range(n))
    ax.set_yticklabels(top['feature'].values[::-1], fontsize=9)
    ax.set_xlabel('Importancia'); ax.set_title(f'Top {n} Features — RF', fontweight='bold')
    legend_el = [mpatches.Patch(facecolor=v, label=k) for k,v in COLOR_MAP.items()]
    ax.legend(handles=legend_el, loc='lower right', fontsize=9)
    ax.spines[['top','right']].set_visible(False); ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    return fig

# ── Análisis GridSearch ───────────────────────────────────────────────────────
def show_gridsearch(model_choice):
    if model_choice == 'Random Forest':
        res = pd.DataFrame(rf_grid.cv_results_)
        pivot = res.pivot_table(index='param_n_estimators',
                                columns='param_max_depth',
                                values='mean_test_score')
    else:
        res = pd.DataFrame(gb_grid.cv_results_)
        pivot = res.pivot_table(index='param_n_estimators',
                                columns='param_learning_rate',
                                values='mean_test_score')
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
                linewidths=0.5, cbar_kws={'label':'F1 CV'})
    ax.set_title(f'GridSearch — {model_choice}\n(F1 Cross-Validation 5-fold)', fontweight='bold')
    plt.tight_layout()
    return fig

print('✅ Funciones del dashboard definidas')

In [ ]:
with gr.Blocks(
    title='Twitter Gender Classifier — Fase 2 AM',
    theme=gr.themes.Soft(primary_hue='orange', secondary_hue='teal')
) as demo:

    gr.Markdown("""
    # 🐦 Twitter Gender Classifier — Fase 2: Aprendizaje de Máquina
    **SVM · Random Forest · Gradient Boosting · Stacking** | Javeriana · Sistemas Inteligentes
    """)

    with gr.Tabs():

        # Tab 1: Clasificar tweet
        with gr.Tab('🔍 Clasificar Tweet'):
            gr.Markdown('### Predice el género del autor — elige qué modelo usas')
            with gr.Row():
                with gr.Column(scale=1):
                    tweet_in   = gr.Textbox(label='Tweet', placeholder='Escribe un tweet...', lines=3)
                    model_dd   = gr.Dropdown(MODEL_NAMES, value=MODEL_NAMES[-1], label='Modelo')
                    pred_btn   = gr.Button('🚀 Predecir', variant='primary')
                    result_txt = gr.Textbox(label='Resultado', interactive=False)
                    feat_txt   = gr.Textbox(label='Features activados', interactive=False, lines=5)
                with gr.Column(scale=1):
                    proba_plot = gr.Plot(label='Probabilidades')
            pred_btn.click(predict_gender, [tweet_in, model_dd], [result_txt, proba_plot, feat_txt])
            gr.Examples([
                ['Just got my nails done, feeling amazing! 💅✨ #selfcare', 'Stacking (SVM+RF+GB→LR)'],
                ['The match last night was insane!! That last-minute goal 🔥⚽ #UCL', 'Random Forest'],
                ['Finished my book club pick. Beautiful story about grief.', 'SVM (Linear)'],
                ['Just upgraded my PC, RTX 4090 arrived!! #gaming #pcbuild', 'Gradient Boosting'],
            ], inputs=[tweet_in, model_dd])

        # Tab 2: Comparar modelos
        with gr.Tab('📊 Comparar Modelos'):
            gr.Markdown('### Evolución del proyecto — modelos previos vs Fase 2 AM')
            comp_btn  = gr.Button('🔄 Generar comparativa', variant='primary')
            comp_plot = gr.Plot()
            comp_btn.click(show_comparison, [], [comp_plot])
            gr.Markdown('#### Tabla de resultados')
            gr.Dataframe(
                value=df_results[['Modelo','Accuracy','F1-Weighted','AUC-ROC','Fase']],
                interactive=False
            )

        # Tab 3: Explorar features
        with gr.Tab('🔤 Explorar Features'):
            with gr.Row():
                with gr.Column():
                    n_slider = gr.Slider(5, 40, value=20, step=5, label='Top N features')
                    fi_btn   = gr.Button('📈 Feature Importance (RF)', variant='primary')
                    fi_plot  = gr.Plot()
                    fi_btn.click(show_top_features, [n_slider], [fi_plot])
                with gr.Column():
                    g_radio = gr.Radio(['Femenino','Masculino'], value='Femenino', label='Género')
                    wc_btn  = gr.Button('☁️ WordCloud', variant='primary')
                    wc_plot = gr.Plot()
                    wc_btn.click(show_wordcloud, [g_radio], [wc_plot])

        # Tab 4: Análisis experimental GridSearch
        with gr.Tab('🧪 Protocolo Experimental'):
            gr.Markdown('### Efecto de hiperparámetros — resultados del GridSearch')
            gs_dd  = gr.Dropdown(['Random Forest','Gradient Boosting'], value='Random Forest',
                                  label='Modelo')
            gs_btn = gr.Button('🔬 Mostrar heatmap CV', variant='primary')
            gs_plt = gr.Plot()
            gs_btn.click(show_gridsearch, [gs_dd], [gs_plt])

demo.launch(inline=True, share=False)